<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第3章：习题解答

本 Notebook 中使用的依赖包：

In [2]:
%pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt  # 安装项目所需的所有依赖
from importlib.metadata import version  # 从标准库获取已安装库的版本信息

used_libraries = [                      # 待检查版本的库列表
    "reasoning_from_scratch",           # 本项目主库
    "torch",                            # PyTorch
    "tokenizers"                        # 被 reasoning_from_scratch 依赖的分词器库
]

for lib in used_libraries:              # 逐个库查询并输出版本
    print(f"{lib} version: {version(lib)}")  # 使用 version 函数获取并打印版本号


reasoning_from_scratch version: 0.1.10
torch version: 2.9.0+cu126
tokenizers version: 0.22.1


&nbsp;
## 练习 3.1：补充更多测试用例

- 可添加的测试案例理论上无穷无尽。
- 下方列出了一些较有意思的示例。

In [3]:
from reasoning_from_scratch.ch03 import (  # 从 ch03 模块导入演示表格函数
    run_demos_table
)

more_tests = [
    # 不同括号类型的等价性
    ("check_16", "[1, 2]", "(1, 2)", True),

    # 科学计数法等价
    ("check_17", "1e-3", "0.001", True),

    # 代数化简，含幂次符号
    ("check_18", "(-3)^2", "9", True),

    # Unicode 减号（U+2212）与 ASCII 连字符减号的等价性
    ("check_19", "−1", "-1", True),
]

run_demos_table(more_tests)  # 运行测试并打印结果表


Test     | Expect | Got   | Status
check_16 | True   | True  | PASS  
check_17 | True   | True  | PASS  
check_18 | True   | True  | PASS  
check_19 | True   | False | FAIL  

Passed 3/4


- 可以看到，除了 `check_19` 之外，其余测试都通过；`check_19` 把普通减号换成了肉眼几乎分辨不出的 Unicode 减号。
- 只需在 `normalize_text` 内加上一行映射，就能修复这一用例，如把 Unicode 减号统一替换为普通的 -。

```python
text = text.replace("−", "-")
# or
text = text.replace("\u2212", "-")
```

- 还有一个看起来很有意思的测试用例如下：

In [4]:
extra_tests_1 = [
    ("check_20", "Text around answer 3.", "3", True)  # 文字包裹答案的情况，期望判断为等价
]

run_demos_table(extra_tests_1)  # 运行额外测试


Test     | Expect | Got   | Status
check_20 | True   | False | FAIL  

Passed 0/1


- 乍看之下，会觉得我们的代码无法处理这种带文字的用例，其实这是个设计不佳的测试。
- 实际上，`run_demos_table` 只用于测试 `grade_answer` 函数，不多也不少。
- `grade_answer` 从不会直接看到整段答复，而是在调用前就已经从文本中提取出纯粹的答案片段。

换句话说，如果想测试带文本的答案，就需要像下面这样调用测试：

In [ ]:
from reasoning_from_scratch.ch03 import (  # 从 ch03 模块导入答案提取函数
    extract_final_candidate
)

extra_tests_2 = [
    ("check_20",
     extract_final_candidate("Text around answer 3."),  # 先从文本中提取最终候选答案
     "3", True)
]
run_demos_table(extra_tests_2)  # 运行测试并打印结果表


Test     | Expect | Got  | Status
check_20 | True   | True | PASS  

Passed 1/1


&nbsp;
## 练习 3.2：计算平均回复长度

- 方案 A：可以在 `evaluate_math500_stream` 函数中加入以下代码：

```python
# ...
# below `num_correct = 0`
total_len = 0

# ...
# inside for i, row in enumerate(math_data, start=1):
# anywhere below `gen_text = ...`
total_len += len(tokenizer.encode(gen_text))

# ...
# anywhere at the bottom before the return statement
avg_len = total_len / num_examples
print(f"Average length: {avg_len:.2f} tokens")
```

- 另一种做法是直接利用主章节运行 `evaluate_math500_stream` 时生成的 `.jsonl` 文件来统计回复长度。
- 首先像下面这样读取 `.jsonl`：

In [9]:
import json
from pathlib import Path

WHICH_MODEL = "base"  # 指定模型类型
device="cuda"
# 需根据实际文件名调整：默认使用 base 模型在 mps 设备上的结果文件
local_path = Path(f"math500-{device}.jsonl")
if not local_path.exists():  # 若文件不存在则提示先运行主笔记生成
    raise FileNotFoundError(
        f"{local_path} not found. Run ch03_main.ipynb to create it."
    )

results = []  # 存放读取的结果记录
with open(local_path, "r") as f:
    for line in f:
        if line.strip():  # 跳过空行
            results.append(json.loads(line))  # 逐行解析 JSON 记录并加入列表

print("Number of entries:", len(results))  # 输出读取到的条目数量


Number of entries: 10


- 每条记录里有多个字段，不过我们只关心 `"generated_text"` —— 它保存了模型的完整答案：

In [ ]:
print(results[0].keys())  # 打印首条结果记录的所有键名


dict_keys(['index', 'problem', 'gtruth_answer', 'generated_text', 'extracted', 'correct'])


- 每条记录虽然包含多个字段，但我们只需要 `"generated_text"`，也就是模型完整回答所在的键：

In [ ]:
from reasoning_from_scratch.qwen3 import (         # 导入下载函数和 Qwen3 分词器
    download_qwen3_small,
    Qwen3Tokenizer
)

if WHICH_MODEL == "base":                          # 基础版模型分支

    download_qwen3_small(
        kind="base", tokenizer_only=True, out_dir="qwen3"  # 仅下载基础版分词器
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json" # 基础版分词器路径
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 创建分词器实例

elif WHICH_MODEL == "reasoning":                   # reasoning 版模型分支

    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir="qwen3"      # 仅下载 reasoning 版分词器
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"     # reasoning 版分词器路径
    tokenizer = Qwen3Tokenizer(                                    # 创建分词器实例并启用聊天模板与思维链标志
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )


- 接着按下列方式计算平均长度，和直接修改 `evaluate_math500_stream` 得到的效果类似：

In [16]:
total_len = 0                                      # 累计所有样本的 token 总数

for item in results:                               # 遍历每条生成结果
    num_tokens = len(tokenizer.encode(item["generated_text"]))  # 对生成文本做分词后计数
    total_len += num_tokens                        # 累加 token 数

avg_len = total_len / len(results)                 # 计算平均 token 长度
print(f"Average length: {avg_len:.2f} tokens")     # 打印平均长度


Average length: 95.60 tokens


| Mode      | Device  | Average length | MATH-500 size  |
|-----------|---------|----------------|----------------|
| Base      | CPU     | 97.3           | 10             |
| Base      | MPS     | 98.0           | 10             |
| Reasoning | CPU     | 891.80         | 10             |
| Reasoning | MPS     | 1159.30        | 10             |
|           |         |                |                |
| Base      | CUDA    | 96.74          | 500            |
| Reasoning | CUDA    | 1361.21        | 500            |


- 可以看到，正如预期，推理版模型的回答明显更长。

&nbsp;
## 练习 3.3：扩展或替换评估数据集

- 如果想在更大的数据集上评估，只需把 `math_data[:10]` 换成其他切片或更大的数量（最高 500）。

```python
num_correct, num_examples, acc = evaluate_math500_stream(  # 运行 MATH-500 评测
    model, tokenizer, device, 
    math_data=math_data[:10],          # 仅使用前 10 条样本做快速测试
    max_new_tokens=2048,               # 单条生成的最大 token 数
    verbose=False                      # 不打印详细生成过程
)

```

- The table below shows the accuracy values for different dataset sizes (since the MATH-500 test set is already shuffled, no additional shuffling was applied)

| Mode      | Device  | Accuracy | MATH-500 size  |
|-----------|---------|----------|----------------|
| Base      | CUDA    | 30.0%    | 10             |
| Base      | CUDA    | 34.0%    | 50             |
| Base      | CUDA    | 27.0%    | 100            |
| Base      | CUDA    | 31.0%    | 200            |
| Base      | CUDA    | 15.3%    | 500            |
|           |         |          |                |
| Reasoning | CUDA    | 90.0%    | 10             |
| Reasoning | CUDA    | 58.0%    | 50             |
| Reasoning | CUDA    | 58.0%    | 100            |
| Reasoning | CUDA    | 56.0%    | 200            |
| Reasoning | CUDA    | 48.2%    | 500            |

- 上述结果可以看出，仅用前 10 个样本并不能代表在完整 500 道 MATH-500 题目上的表现。

- 另外，我们还可以仿照 MATH-500 自行构建一套新数据集。
- 本仓库已经附带一个 MATH-500 风格的数据集，只需在本章主代码里把文件名 `math500_test.json` 改成  `math_new50_exercise.json` (位于https://github.com/rasbt/reasoning-from-scratch/tree/main/ch03/01_main-chapter-code)
- 该数据集上基础版和推理版的表现为：
    - base：36.0%（18/50）
    - reasoning：80.0%（40/50）
- 这说明尽管 MATH-500 测试集可能在 Qwen3 训练时出现过，但模型在新题上依然保持类似表现，说明并未对原始 MATH-500 数据过度拟合。

&nbsp;
## 练习 3.4：尝试不同提示模板

- 我们可以改用章节中提到的另一种提示语，把“Question”替换为“Problem”，大致如下：

```python
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"        # 角色与任务说明
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"                         # 要求用 \boxed{} 输出最终答案
        f"Problem:\n{prompt}\n\nAnswer:"             # 插入题目并提示开始作答
    )
    return template                                  # 返回完整提示词模板

```

- 采用该提示后，基础模型在 500 个样本上的准确率从 15.3% 提升到 31.2%。
- 相反，推理模型的性能从 50.8% 略降到 50.0%。
- 说明基础模型对提示格式极度敏感（可能在训练中记住了某些特定格式的 MATH-500 样本），而推理模型几乎不受影响。